# 02 — Extractor evaluation

Runs the call extractor over the labeled gold set in `data/gold/extraction_gold.jsonl`
and reports call-detection precision/recall + per-field accuracy.

Requires `ANTHROPIC_API_KEY` in `.env` to actually run extraction. The notebook is
structured so it loads the gold set + report shape even without an API key, so you
can inspect the eval scaffolding offline.

**Scope**: gold set is currently ~20 bootstrap examples. The target is ~200 hand-labeled
segments — grow this file as we encounter real-world failure modes.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from app.config import load_env
from app.extract.eval import load_gold

## Load gold set

In [ ]:
gold = load_gold()
print(f'loaded {len(gold)} gold items')
print(f'  positives: {sum(1 for g in gold if g.expected is not None)}')
print(f'  negatives: {sum(1 for g in gold if g.expected is None)}')

df_gold = pd.DataFrame([
    {'id': g.id, 'is_call': g.expected is not None, 'preview': g.source_text[:80] + '...', 'notes': g.notes}
    for g in gold
])
df_gold

## Run extractor (requires ANTHROPIC_API_KEY)

In [ ]:
env = load_env()
if not env.anthropic_api_key:
    print('ANTHROPIC_API_KEY not set — skipping live extraction.')
    print('Set it in .env, restart the kernel, and re-run this cell.')
    report = None
else:
    from app.extract.eval import evaluate
    from app.extract.llm_extractor import LLMExtractor
    extractor = LLMExtractor()
    report = evaluate(extractor, gold=gold)
    print('done')

## Summary

In [ ]:
if report is None:
    print('no report (set ANTHROPIC_API_KEY first)')
else:
    pd.DataFrame(report.summary_table(), columns=['metric', 'value'])

## Per-item details (errors first)

In [ ]:
if report is None:
    pd.DataFrame()
else:
    rows = []
    for it in report.per_item:
        is_error = it['gold_is_call'] != it['pred_is_call']
        rows.append({
            'id': it['id'],
            'error': is_error,
            'gold': '+' if it['gold_is_call'] else '-',
            'pred': '+' if it['pred_is_call'] else '-',
            'pred_ticker': (it['predicted'] or {}).get('ticker'),
            'pred_dir': (it['predicted'] or {}).get('direction'),
            'pred_entry_type': (it['predicted'] or {}).get('entry_type'),
            'notes': it['notes'][:60],
        })
    df = pd.DataFrame(rows).sort_values('error', ascending=False)
    df